# Optimization Strategies Implementation on Data


This notebook focuses on implementing and analyzing various optimization strategies for maintenance decision-making. The primary goal is to evaluate **Cost Minimization** and **Revenue Maximization** strategies under different scenarios using input data, exploratory data analysis (EDA), and derived features for features engineering. 

1. **Data Integration and EDA**:
   - The notebook incorporates data from various sources and performs exploratory data analysis to understand fault distributions, seasonality, operational metrics, and revenue implications.
   - Additional features are created from raw data to support decision-making processes.

2. **Class-Based Framework**:
   - Multiple classes are defined to encapsulate different strategies for maintenance optimization:
     - **MaintenanceStrategies_cost**: Focuses on minimizing costs by comparing fixed internal maintenance costs against variable external maintenance costs.
     - **MaintenanceStrategy_LinearOpt_cost**: Uses an optimization framework (linear programming) to minimize costs under similar parameters.
     - **MaintenanceStrategies_revenue**: Maximizes revenue by evaluating the trade-offs between repair times, downtime, and operational revenue.

3. **Scenario Analysis**:
   - The strategies are evaluated under various conditions, including:
     - **Without Preventative Maintenance**: Full fault counts are considered.
     - **With Preventative Maintenance**: Fault counts are reduced by a specified percentage (e.g., 60%) to simulate the impact of preventative measures.
   - The decisions for internal vs. external maintenance are analyzed for each fault type.

4. **Results and Insights**:
   - The notebook provides comparative analyses of the strategies, highlighting consistent patterns in decision-making across different methodologies.
   - Key metrics such as total cost, revenue, downtime, and maintenance allocation are computed and visualized.

5. **Data-Driven Decision-Making**:
   - All analyses are data-driven, leveraging derived features and EDA to ensure the strategies are grounded in realistic operational scenarios.
   - The results demonstrate how inputs from the data can inform optimal maintenance strategies, balancing cost efficiency and revenue potential.

This comprehensive framework enables an in-depth evaluation of maintenance strategies, providing actionable insights into internal vs. external maintenance decisions under various operational conditions.


In [1]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


import sys
import os

# Get the path to the src folder
src_path = os.path.abspath(os.path.join(os.getcwd(), "../src"))
if src_path not in sys.path:
    sys.path.append(src_path)


from optimization import MaintenanceStrategies_cost
from optimization import MaintenanceStrategy_LinearOpt_cost
from optimization import MaintenanceStrategies_revenue


## Maintenance Strategy 1 (Minimize Maintenance Cost)

## **Using a Custom Class called MaintenanceStrategies_cost**

The `MaintenanceStrategies_cost` class is designed to determine the optimal maintenance strategy for minimizing total maintenance costs based on internal and external maintenance options. It incorporates factors such as fault rates, seasonal demand variations, and preventive maintenance effectiveness.

## Key Features

### 1. Initialization (`__init__` Method)
- **Parameters**:
  - `max_faults`: Maximum number of faults expected to occur.
  - `high_seasonality`: Proportion of faults occurring during the high-demand season (e.g., 0.7 indicates 70% faults happen in the high season).
  - `cost_internal`: Fixed cost for handling all faults internally.
  - `cost_external_normal`: Cost per fault for external maintenance during normal demand.
  - `cost_external_high_demand`: Cost per fault for external maintenance during high-demand seasons.
  - `prevention_effect`: Effectiveness of preventive maintenance in reducing fault occurrences.

- **Fault Distribution**:
  - Calculates the reduced number of faults after applying preventive measures (`faults_to_address`).
  - Splits faults into high-demand and normal seasons using the `high_seasonality` parameter.

### 2. Decision-Making (`maintenance_decsion` Method)
- **Cost Calculation**:
  - Compares the total cost of addressing faults externally (`total_external_cost`) with the fixed cost of internal maintenance (`cost_internal`).
  
- **Strategy Selection**:
  - If external maintenance is cheaper, faults are assigned to external providers, split across normal and high-demand seasons.
  - If internal maintenance is more cost-effective, all faults are addressed internally.

### 3. Outputs
- Returns the following:
  - `total_cost`: Total cost of the chosen maintenance strategy.
  - `total_internal_maintenance`: Number of faults addressed internally.
  - `total_external_normal_maintenance`: Number of faults addressed externally during the normal season.
  - `total_external_high_maintenance`: Number of faults addressed externally during the high-demand season.

### Without Any Preventation Maintenance

## Retrieving Necessary inputs for the model from the Data and EDA Analysis

#### Occurrences of Each Fault Type (Gotten from Exploratory Fault Type Analysis.ipynb)
**Types of faults and their total Faults in the Year 2014**

- **GF(Generator Heating Fault):** Indicates overheating or related issues in the generator, which could suggest an overload or poor cooling in the generator unit.
    - *Total Faults in the Year 2014: **43***

- **MF (Mains Failure Fault):** Reflects a failure in the main power supply, likely causing interruptions or necessitating backup power.
    - *Total Faults in the Year 2014: **20***

- **FF (Feeding Fault):** Relates to issues in the feeding mechanism, which might impact fuel or resource input, affecting the system’s operation.
    - *Total Faults in the Year 2014: **254***

- **AF (Air Cooling Fault):** Points to problems with air cooling, likely resulting in increased temperatures if the cooling system is ineffective.
    - *Total Faults in the Year 2014: **62***

- **EF (Excitation Fault):** would typically relate to issues in the generator's excitation system, affecting voltage regulation.
    - *Total Faults in the Year 2014: **174***

In [5]:
# Count the occurrences of each fault type
total_fault_counts = {'FF':254, 'EF':174, 'AF':62, 'GF':43, 'MF':20}

Percentage of **High Seasons** throughout the year (Gotten from Season, Production and Revenue Anaylsis.ipynb)

$$
\text{Percentage of High Seasons} = \frac{29}{50} \approx 58.00\%
$$

This insight highlights that High Season months account for approximately 58% of the year, indicating a significant period of heightened market activity. This information can guide resource allocation, marketing strategies, and operational planning during these critical months. **This will be as varaible called high_seasonality=0.4167 for optimization to decide on Maintenance Strategies for determine external vs internal.**

In [6]:
high_seasonality=0.58

In [13]:
# Define cost parameters (given from Case Study)
cost_internal = 750000
cost_external_normal = 50000
cost_external_high_demand = 150000
prevention_effect = 0 #Analysis results without any pevention effects. 

# Dictionary to store results
results = []

# Iterate through each fault type in total_fault_counts
for fault_type, max_faults in total_fault_counts.items():
    # Initialize the MaintenanceStrategies_cost class
    maintenance_strategy = MaintenanceStrategies_cost(
        max_faults=max_faults,
        high_seasonality=high_seasonality,
        cost_internal=cost_internal,
        cost_external_normal=cost_external_normal,
        cost_external_high_demand=cost_external_high_demand,
        prevention_effect=prevention_effect
    )
    
    # Run the maintenance decision method
    total_cost, total_internal_maintenance, total_external_normal_maintenance, total_external_high_maintenance = maintenance_strategy.maintenance_decsion()
    
    # Determine if internal maintenance was chosen (yes/no)
    internal_maintenance_decision = "Yes" if total_internal_maintenance > 0 else "No"
    
    # Store the results for this fault type
    results.append({
        "Fault Type": fault_type,
        "Max Faults": max_faults,
        "Total Cost": total_cost,
        "Internal Maintenance": total_internal_maintenance,
        "External Normal Maintenance": total_external_normal_maintenance,
        "External High Maintenance": total_external_high_maintenance,
        "Internal Maintenance Decision": internal_maintenance_decision
    })

# Convert the results to a DataFrame for easy visualization
results_df = pd.DataFrame(results)


results_df


,Fault Type,Max Faults,Total Cost,Internal Maintenance,External Normal Maintenance,External High Maintenance,Internal Maintenance Decision
0,FF,254,750000,254,0,0,Yes
1,EF,174,750000,174,0,0,Yes
2,AF,62,750000,62,0,0,Yes
3,GF,43,750000,43,0,0,Yes
4,MF,20,750000,20,0,0,Yes




1. **Internal Maintenance Decision**:
   - For all fault types (`FF`, `EF`, `AF`, `GF`, and `MF`), the maintenance strategy selected **internal maintenance** as the optimal choice.
   - The result indicates that the fixed cost of internal maintenance (`750,000`) was consistently more cost-effective than the combined costs of external maintenance, even without any prevention effects applied.

2. **Uniform Costs**:
   - The **Total Cost** is constant at `750,000` for all fault types, demonstrating the fixed nature of internal maintenance costs.
   - The absence of prevention effects (`prevention_effect = 0`) ensures that all faults are addressed without any reduction, reinforcing the cost-efficiency of internal maintenance.


The analysis demonstrates that **internal maintenance is the preferred strategy for all fault types** under the given conditions. The fixed cost of internal maintenance (`750,000`) proved more advantageous than the variable costs of external maintenance, irrespective of fault distribution. This finding highlights the financial efficiency and reliability of internal maintenance for addressing all fault scenarios.


## **Using a Linear Solve Class called MaintenanceStrategy_LinearOpt_cost**


The `MaintenanceStrategy_LinearOpt_cost` class is a streamlined implementation of the `MaintenanceStrategies_cost` class. It is designed to **minimize maintenance costs** using a linear optimization approach with Pyomo and the Gurobi solver. This class determines the optimal maintenance strategy (internal vs. external) based on cost considerations.

---

#### Key Features

### 1. Inputs
- **Fault Parameters**:
  - `max_faults`: Total number of faults.
  - `high_seasonality`: Proportion of faults occurring during high-demand seasons.
- **Cost Parameters**:
  - `cost_internal`: Fixed cost for internal maintenance.
  - `cost_external_normal`: Cost per fault for external maintenance during normal seasons.
  - `cost_external_high_demand`: Cost per fault for external maintenance during high-demand seasons.
- **Prevention Effect**:
  - Reduces the fault burden by a specified proportion (`prevention_effect`).

### 2. Optimization Model
- **Decision Variable**:
  - `x_internal`: A binary variable indicating whether internal maintenance is chosen (`1`) or not (`0`).
- **Objective Function**:
  - Minimizes the total cost of handling faults, which includes:
    - External maintenance costs for faults during normal and high-demand seasons.
    - Internal maintenance fixed costs if chosen.
- **Seasonal Fault Distribution**:
  - Splits faults into high-demand and normal seasons to calculate external maintenance costs.

### 3. Solver
- Uses the Gurobi solver to determine the optimal maintenance decision that minimizes total costs.

### 4. Output
- A dictionary summarizing the results:
  - Faults handled internally and externally.
  - Maintenance costs for each strategy.
  - Total cost of the optimal strategy.



---

## Differences Between the Classes

| **Aspect**                | **`MaintenanceStrategies_cost`**             | **`MaintenanceStrategy_LinearOpt_cost`**           |
|---------------------------|----------------------------------------------|---------------------------------------------------|
| **Implementation**         | Direct cost comparison.                     | Uses linear optimization with Pyomo and Gurobi.  |
| **Approach**               | Manual comparison of costs for decisions.   | Automated optimization framework.                |
| **Conciseness**            | Requires more manual logic for calculations.| More concise and computationally efficient.      |
| **Solver Integration**     | Does not use an optimization solver.        | Explicitly integrates Gurobi for scalability.    |

---

## Summary

The `MaintenanceStrategy_LinearOpt_cost` class performs the same function as `MaintenanceStrategies_cost` but uses a linear optimization approach for simplicity and efficiency. Both classes provide the same decision-making framework for minimizing maintenance costs, but the linear optimization method is better suited for larger datasets and complex scenarios.


In [ ]:
# Define cost parameters (given from Case Study)
cost_internal = 750000
cost_external_normal = 50000
cost_external_high_demand = 150000
prevention_effect = 0 #Analysis results without any pevention effects. 

# Dictionary to store results
linear_opt_results = []

# Iterate through each fault type in total_fault_counts
for fault_type, max_faults in total_fault_counts.items():
    # Initialize the MaintenanceStrategy_LinearOpt_cost class
    maintenance_strategy = MaintenanceStrategy_LinearOpt_cost(
        max_faults=max_faults,
        high_seasonality=high_seasonality,
        cost_internal=cost_internal,
        cost_external_normal=cost_external_normal,
        cost_external_high_demand=cost_external_high_demand,
        prevention_effect=prevention_effect
    )
    
    # Solve the optimization problem
    results = maintenance_strategy.solve()
    
    # Determine if internal maintenance was chosen (yes/no)
    internal_maintenance_decision = "Yes" if results["Internal Maintenance"] > 0 else "No"
    
    # Append results for this fault type
    linear_opt_results.append({
        "Fault Type": fault_type,
        "Max Faults": max_faults,
        "Total Cost": results["Total Cost"],
        "Internal Maintenance": results["Internal Maintenance"],
        "External Maintenance (Normal)": results["External Maintenance (Normal)"],
        "External Maintenance (High-Demand)": results["External Maintenance (High-Demand)"],
        "Internal Maintenance Decision": internal_maintenance_decision
    })

# Convert the results to a DataFrame for easy visualization
linear_opt_results_df = pd.DataFrame(linear_opt_results)

linear_opt_results_df

Read LP format model from file /var/folders/f1/tcfth2f16mv8s95myjjfx84h0000gn/T/tmp2xeuy89d.pyomo.lp
Reading time = 0.00 seconds
x1: 1 rows, 2 columns, 1 nonzeros
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (mac64[arm] - Darwin 22.6.0 22H313)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 1 rows, 2 columns and 1 nonzeros
Model fingerprint: 0xc405c1b2
Variable types: 1 continuous, 1 integer (1 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [3e+07, 3e+07]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 750000.00000

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 1: 750000 

Optimal solution found (tolerance 1.00e-04)
Best objective 7.500000000000e+05, best bound 7.500000000000e+05, gap 0.0000%
Read LP format model from file /var/f

,Fault Type,Max Faults,Total Cost,Internal Maintenance,External Maintenance (Normal),External Maintenance (High-Demand),Internal Maintenance Decision
0,FF,254,750000.0,254.0,0.0,0.0,Yes
1,EF,174,750000.0,174.0,0.0,0.0,Yes
2,AF,62,750000.0,62.0,0.0,0.0,Yes
3,GF,43,750000.0,43.0,0.0,0.0,Yes
4,MF,20,750000.0,20.0,0.0,0.0,Yes



The `MaintenanceStrategy_LinearOpt_cost` class is designed to optimize the selection between internal and external maintenance using a linear programming approach. Here's a concise explanation of its functionality and how it compares to the `MaintenanceStrategies_cost` class.


**Same Results**:
   - For the provided scenarios, both classes yield identical results: **internal maintenance is selected for all fault types** due to its lower cost compared to external maintenance.


**Approach**:
     - `MaintenanceStrategies_cost`: Uses a rule-based comparison of fixed and variable costs to make decisions.
     - `MaintenanceStrategy_LinearOpt_cost`: Uses an **optimization framework** (linear programming) for decision-making.


The `MaintenanceStrategy_LinearOpt_cost` class achieves the same outcomes as `MaintenanceStrategies_cost` for situation without any perventative maintenance. However, its optimization framework offers greater flexibility for more complex problems. For straightforward scenarios, both approaches are functionally equivalent, but `MaintenanceStrategy_LinearOpt_cost` is better suited for situations requiring additional constraints or variables.


### With Preventation Maintenance (Resulting in 60% reduction in Faults)

## **Using a Custom Class called MaintenanceStrategies_cost**

In [15]:
# Define cost parameters (given from Case Study)
cost_internal = 750000
cost_external_normal = 50000
cost_external_high_demand = 150000
prevention_effect = 0.6 #Analysis results with pevention effects. 

# Dictionary to store results
results = []

# Iterate through each fault type in total_fault_counts
for fault_type, max_faults in total_fault_counts.items():
    # Initialize the MaintenanceStrategies_cost class
    maintenance_strategy = MaintenanceStrategies_cost(
        max_faults=max_faults,
        high_seasonality=high_seasonality,
        cost_internal=cost_internal,
        cost_external_normal=cost_external_normal,
        cost_external_high_demand=cost_external_high_demand,
        prevention_effect=prevention_effect
    )
    
    # Run the maintenance decision method
    total_cost, total_internal_maintenance, total_external_normal_maintenance, total_external_high_maintenance = maintenance_strategy.maintenance_decsion()
    
    # Determine if internal maintenance was chosen (yes/no)
    internal_maintenance_decision = "Yes" if total_internal_maintenance > 0 else "No"
    
    # Store the results for this fault type
    results.append({
        "Fault Type": fault_type,
        "Max Faults": max_faults,
        "Total Cost": total_cost,
        "Internal Maintenance": total_internal_maintenance,
        "External Normal Maintenance": total_external_normal_maintenance,
        "External High Maintenance": total_external_high_maintenance,
        "Internal Maintenance Decision": internal_maintenance_decision
    })

# Convert the results to a DataFrame for easy visualization
results_df = pd.DataFrame(results)


results_df


,Fault Type,Max Faults,Total Cost,Internal Maintenance,External Normal Maintenance,External High Maintenance,Internal Maintenance Decision
0,FF,254,750000,102,0,0,Yes
1,EF,174,750000,70,0,0,Yes
2,AF,62,750000,25,0,0,Yes
3,GF,43,750000,17,0,0,Yes
4,MF,20,750000,8,0,0,Yes




1. **Internal Maintenance Decision**:
   - Similar to the results without preventative maintenance, **all fault types** (`FF`, `EF`, `AF`, `GF`, and `MF`) resulted in an **Internal Maintenance Decision** ("Yes").
   - This indicates that even with a 60% reduction in faults, internal maintenance remains the most cost-effective strategy due to its fixed cost structure.

2. **Impact of Preventative Maintenance**:
   - The introduction of preventative maintenance significantly reduced the **Internal Maintenance** load:
     - For example, the `FF` fault type saw a reduction from **254 faults** (without preventative maintenance) to **102 faults** (with preventative maintenance).
     - Similar reductions are observed across all fault types.

3. **Total Cost**:
   - The **Total Cost** remains constant at `750,000` across all fault types, as internal maintenance has a fixed cost regardless of the fault count.



#### Comparison with Results Without Preventative Maintenance:
| Metric                        | Without Preventative Maintenance | With Preventative Maintenance (60%) |
|-------------------------------|-----------------------------------|-------------------------------------|
| Total Cost                    | 750,000                         | 750,000                             |
| Internal Maintenance (FF)     | 254                             | 102                                 |
| Internal Maintenance (EF)     | 174                             | 70                                  |
| Internal Maintenance (AF)     | 62                              | 25                                  |
| Internal Maintenance (GF)     | 43                              | 17                                  |
| Internal Maintenance (MF)     | 20                              | 8                                   |
| External Normal Maintenance   | 0                               | 0                                   |
| External High Maintenance     | 0                               | 0                                   |
| Internal Maintenance Decision | Yes                             | Yes                                 |


Preventative maintenance successfully reduced the number of faults by 60% across all fault types, significantly decreasing the maintenance workload while maintaining the total cost at `750,000` due to the fixed cost structure of internal maintenance. The strategy remained consistent, with **all faults being addressed via internal maintenance**, highlighting the resilience and cost-effectiveness of internal maintenance under varying fault conditions.


## **Using a Linear Solve Class called MaintenanceStrategy_LinearOpt_cost**

In [ ]:
# Define cost parameters (given from Case Study)
cost_internal = 750000
cost_external_normal = 50000
cost_external_high_demand = 150000
prevention_effect = 0.6 #Analysis results with pevention effects. 

# Dictionary to store results
linear_opt_results = []

# Iterate through each fault type in total_fault_counts
for fault_type, max_faults in total_fault_counts.items():
    # Initialize the MaintenanceStrategy_LinearOpt_cost class
    maintenance_strategy = MaintenanceStrategy_LinearOpt_cost(
        max_faults=max_faults,
        high_seasonality=high_seasonality,
        cost_internal=cost_internal,
        cost_external_normal=cost_external_normal,
        cost_external_high_demand=cost_external_high_demand,
        prevention_effect=prevention_effect
    )
    
    # Solve the optimization problem
    results = maintenance_strategy.solve()
    
    # Determine if internal maintenance was chosen (yes/no)
    internal_maintenance_decision = "Yes" if results["Internal Maintenance"] > 0 else "No"
    
    # Append results for this fault type
    linear_opt_results.append({
        "Fault Type": fault_type,
        "Max Faults": max_faults,
        "Total Cost": results["Total Cost"],
        "Internal Maintenance": results["Internal Maintenance"],
        "External Maintenance (Normal)": results["External Maintenance (Normal)"],
        "External Maintenance (High-Demand)": results["External Maintenance (High-Demand)"],
        "Internal Maintenance Decision": internal_maintenance_decision
    })

# Convert the results to a DataFrame for easy visualization
linear_opt_results_df = pd.DataFrame(linear_opt_results)

linear_opt_results_df

Read LP format model from file /var/folders/f1/tcfth2f16mv8s95myjjfx84h0000gn/T/tmphwnnbr_f.pyomo.lp
Reading time = 0.00 seconds
x1: 1 rows, 2 columns, 1 nonzeros
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (mac64[arm] - Darwin 22.6.0 22H313)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 1 rows, 2 columns and 1 nonzeros
Model fingerprint: 0xec3657ac
Variable types: 1 continuous, 1 integer (1 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+07, 1e+07]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 750000.00000

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 1: 750000 

Optimal solution found (tolerance 1.00e-04)
Best objective 7.500000000000e+05, best bound 7.500000000000e+05, gap 0.0000%
Read LP format model from file /var/f

,Fault Type,Max Faults,Total Cost,Internal Maintenance,External Maintenance (Normal),External Maintenance (High-Demand),Internal Maintenance Decision
0,FF,254,750000.0,102.0,0.0,0.0,Yes
1,EF,174,750000.0,70.0,0.0,0.0,Yes
2,AF,62,750000.0,25.0,0.0,0.0,Yes
3,GF,43,750000.0,17.0,0.0,0.0,Yes
4,MF,20,750000.0,8.0,0.0,0.0,Yes


The `MaintenanceStrategy_LinearOpt_cost` class achieves the same outcomes as `MaintenanceStrategies_cost` for situation with perventative maintenance (60% reduction in faults). In all scenarios (across different fault types), both classes consistently select **internal maintenance** as the optimal strategy. This uniformity demonstrates that regardless of the method—rule-based evaluation or optimization—the decision aligns under the same cost parameters.The converging evidence provided by `MaintenanceStrategies_cost` and `MaintenanceStrategy_LinearOpt_cost` strengthens the reliability of the maintenance decisions. This alignment highlights that both methods effectively address the problem under the same assumptions and parameters, ensuring robust and consistent maintenance strategies. For decision-makers, this convergence underscores the confidence in choosing internal maintenance when cost minimization is the primary objective.







## Maintenance Strategy 2 (Maximize Revenue)


The `MaintenanceStrategies_revenue` class focuses on **maximizing revenue** while managing faults through either internal or external maintenance strategies. It takes into account maintenance costs, downtime, and the impact of preventive maintenance on reducing faults. The class calculates the total revenue generated based on the maintenance strategy and selects the option that results in the **highest revenue**.

---


### 1. Inputs and Parameters
- **Similar to `MaintenanceStrategies_cost`**:
  - `max_faults`: Maximum number of faults.
  - `high_seasonality`: Proportion of faults in high-demand seasons.
  - `cost_internal`: Fixed cost for internal maintenance.
  - `cost_external_normal`: Cost per fault for external maintenance in normal seasons.
  - `prevention_effect`: Reduction in faults due to preventive maintenance.

- **Additional Parameters**:
  - `repair_time_external_hr`: Time (in hours) to repair a fault using external maintenance.
  - `repair_time_internal_hr`: Time (in hours) to repair a fault using internal maintenance.
  - `operation_time_hr`: Total available operation time (in hours).
  - `max_revenue`: Maximum possible revenue achievable within the operation time.

---

### 2. Fault Reduction and Distribution
- **Preventive Maintenance**: Reduces total faults to address (`faults_to_address`) based on `prevention_effect`.
- **Seasonal Distribution**: Divides the remaining faults into high-demand and normal seasons using `high_seasonality`.

---

### 3. Revenue Calculations
- **Revenue for External Maintenance**:
  - Accounts for downtime due to external maintenance (`total_downtime_external_maintenance`) and its associated revenue loss.
  - Deducts external maintenance costs and lost revenue from the maximum possible revenue to compute `revenue_external_maintenance`.

- **Revenue for Internal Maintenance**:
  - Considers downtime caused by internal maintenance (`total_downtime_internal_maintenance`) and its revenue impact.
  - Deducts internal maintenance costs and lost revenue to compute `revenue_internal_maintenance`.

---

### 4. Decision Strategy (`maintenance_decision` Method)
- Compares the revenue generated by internal and external maintenance.
- Chooses the maintenance strategy that results in the **highest total revenue**.

---

## Comparison with `MaintenanceStrategies_cost`

| **Aspect**                       | **`MaintenanceStrategies_cost`**                     | **`MaintenanceStrategies_revenue`**                 |
|-----------------------------------|-----------------------------------------------------|-----------------------------------------------------|
| **Objective**                     | Minimize total maintenance cost.                   | Maximize total revenue.                            |
| **Key Factors in Decision**       | Direct cost comparison between internal and external maintenance. | Considers both maintenance costs and downtime, factoring in revenue loss. |
| **Impact of Downtime**            | Downtime is not explicitly considered.             | Downtime directly affects revenue calculations and decision-making. |
| **Decision Metric**               | Chooses the cheaper option between internal and external maintenance. | Chooses the option that generates the highest revenue. |
| **Output Metrics**                | Total cost, faults addressed by each strategy.      | Total revenue, downtime, cost, and faults handled by each strategy. |
| **Focus**                         | Pure cost minimization.                            | Balances cost, downtime, and revenue generation.    |

---

## Strategy for Deciding Internal vs. External Maintenance

### In `MaintenanceStrategies_cost`:
- **Focus**: Direct comparison of internal and external maintenance costs.
- **Decision**: Chooses the maintenance strategy (internal or external) that minimizes total cost, regardless of downtime or revenue impact.

### In `MaintenanceStrategies_revenue`:
- **Focus**: Balances cost with the impact of downtime on revenue.
- **Decision**:
  - External maintenance is preferred if its lower cost offsets the revenue lost due to its downtime.
  - Internal maintenance is chosen when its higher fixed cost is justified by the reduced downtime and the resulting higher revenue.

---

## Key Differences in Decision-Making

1. **Consideration of Downtime**:
   - `MaintenanceStrategies_revenue` explicitly incorporates downtime and its revenue impact into the decision process, unlike `MaintenanceStrategies_cost`.

2. **Revenue Maximization vs. Cost Minimization**:
   - The `MaintenanceStrategies_cost` class prioritizes minimizing expenses, making it suitable when downtime costs are negligible or fixed.
   - The `MaintenanceStrategies_revenue` class prioritizes maximizing operational revenue, making it suitable when downtime has a significant financial impact.

3. **Complexity of Decision**:
   - `MaintenanceStrategies_cost` decisions are simpler, based solely on cost comparisons.
   - `MaintenanceStrategies_revenue` involves a more nuanced decision-making process, requiring calculations of downtime, operational hours, and revenue loss.




## Retrieving Necessary inputs for the model from the Data and EDA Analysis

#### Reading SCADA data with New Feautres

In [2]:
# Change current directory to parent directory
os.chdir("..")

# Define global variables
RAW_PATH_SCADA_FEATURES = 'data/processed/scada_with_features.csv'


# Attempt to load the file with a try-except block
try:
    # Check if the file exists
    if os.path.exists(RAW_PATH_SCADA_FEATURES):
        # Load the CSV file
        scada_data_features = pd.read_csv(RAW_PATH_SCADA_FEATURES)
        print("File loaded successfully.")
    else:
        raise FileNotFoundError(f"File not found: {RAW_PATH_SCADA_FEATURES}")
except FileNotFoundError as fnf_error:
    print(fnf_error)
except Exception as e:
    print(f"An error occurred while reading the file: {e}")

File loaded successfully.


### Maximum Revenue Generated (New Feature Revenue was added by feature_engineering.py)

This revenue input in the function was generated by the turbines from from April 2014 to April 2015, this revenue is calculated by product of the change in energy (determine through change in WEC: Production kWh: Total energy production in kilowatt-hours, available in SCADA data) with Settlement Point Price from ECROT Market Data 2014 and 2015. This number of $223992786.65 will serve as maximum possible revenue that can be generated by the turbines if there is no downtime or no cost of maintenance due to occuring of faults, a situation where no faults have ocuured. However, if a fault occurs it will require some maintenance which will cause downtime of the turbine and some cost required for performing maintenance, through both of these factors and type of maintenance selected, the revenue will decrease from the maximum possible revenue.

In [ ]:
# Aggregate revenue 
max_generated_revenue = scada_data_features['Revenue'].sum()
# Print the results
print(f"This the revenue generated from April 2014 to April 2015: ${max_generated_revenue:.2f}")

This the revenue generated from April 2014 to April 2015: $223992786.65


### Maximum Possible Operating Hours: Total operating hours of the turbine. (Feature from orignal SCADA data)

This is input in function, Maximum Possible Operating Hours of the tubine. This hours doesn't include time when turbine is not in operation because no wind in the surrounding. This input will help detemine how much revenue is lost due to downtime.  Maximum Revenue Generated divide by the Maximum Operating Hours will provide an estimate of how much revenue is generated in each hour of operation of the turbine. Hence this estimate can be used to determine the loss in revenue, depending duration of downtime in hours. 

In [20]:
# Aggregate revenue 
total_operation_hr = scada_data_features['WEC: Operating Hours'].max()
# Print the results
print(f"This the total operating hours of the turbine from April 2014 to April 2015: {total_operation_hr:.2f} hours")

This the total operating hours of the turbine from April 2014 to April 2015: 7884.00 hours


### Repair Time for Internal and External Maintenance for each fault (This hypothical assumption, not based on any data)

The reason why the repair time for external maintenance is greater than repair time for internal maintenance, is to better evalute the effect of downtime on the decision of maintence to be performed. 

In [21]:
#An Estimate of Repair Time for External Maintenance
repair_time_external_hr=6
#An Estimate of Repair Time for Internal Maintenance
repair_time_internal_hr=8

### Without Any Preventation Maintenance

In [25]:
# Define cost parameters (given from Case Study)
cost_internal = 750000
cost_external_normal = 50000
cost_external_high_demand = 150000
prevention_effect = 0 #Analysis results without any pevention effects. 

# Dictionary to store results
revenue_results = []

# Iterate through each fault type in total_fault_counts
for fault_type, max_faults in total_fault_counts.items():
    # Initialize the MaintenanceStrategies_revenue class
    maintenance_strategy = MaintenanceStrategies_revenue(
        max_faults=max_faults,
        high_seasonality=high_seasonality,
        cost_internal=cost_internal,
        cost_external_normal=cost_external_normal,
        cost_external_high_demand=cost_external_high_demand,
        prevention_effect=prevention_effect,
        repair_time_external_hr=repair_time_external_hr,
        repair_time_internal_hr=repair_time_internal_hr,
        operation_time_hr=total_operation_hr,
        max_revenue=max_generated_revenue
    )
    
    # Run the maintenance decision method
    (
        total_revenue,
        total_downtime,
        total_cost,
        total_internal_maintenance,
        total_external_normal_maintenance,
        total_external_high_maintenance
    ) = maintenance_strategy.maintenance_decision()
    
    # Determine if internal maintenance was chosen (yes/no)
    internal_maintenance_decision = "Yes" if total_internal_maintenance > 0 else "No"
    
    # Append results for this fault type
    revenue_results.append({
        "Fault Type": fault_type,
        "Max Faults": max_faults,
        "Total Revenue": total_revenue,
        "Total Downtime (hrs)": total_downtime,
        "Total Cost": total_cost,
        "Internal Maintenance": total_internal_maintenance,
        "External Maintenance (Normal)": total_external_normal_maintenance,
        "External Maintenance (High-Demand)": total_external_high_maintenance,
        "Internal Maintenance Decision": internal_maintenance_decision
    })

# Convert the results to a DataFrame for easy visualization
revenue_results_df = pd.DataFrame(revenue_results)

# Display the DataFrame
revenue_results_df

,Fault Type,Max Faults,Total Revenue,Total Downtime (hrs),Total Cost,Internal Maintenance,External Maintenance (Normal),External Maintenance (High-Demand),Internal Maintenance Decision
0,FF,254,1.655115e+08,2032,750000,254,0,0,Yes
1,EF,174,1.836946e+08,1392,750000,174,0,0,Yes
2,AF,62,2.091509e+08,496,750000,62,0,0,Yes
3,GF,43,2.134694e+08,344,750000,43,0,0,Yes
4,MF,20,2.186970e+08,160,750000,20,0,0,Yes



1. **Revenue Impact**:
   - The revenue-based strategy focuses on maximizing **Total Revenue** while accounting for downtime caused by faults.

2. **Internal Maintenance Decision**:
   - All fault types (`FF`, `EF`, `AF`, `GF`, `MF`) resulted in an **Internal Maintenance Decision** ("Yes").
   - This indicates that internal maintenance was the optimal strategy for both cost-effectiveness and revenue maximization under the given parameters.



#### Comparative Table:

| Metric                        | MaintenanceStrategies_revenue (No Prevention) | MaintenanceStrategies_cost (No Prevention) |
|-------------------------------|-----------------------------------------------|--------------------------------------------|
| **Focus**                     | Revenue Maximization                          | Cost Minimization                          |
| **Total Revenue**             | Varies by fault type (e.g., `1.655e+08` for `FF`) | Not applicable                             |
| **Total Downtime (hrs)**      | Significant, varies by fault type             | Not considered                             |
| **Total Cost**                | Constant `750,000`                            | Constant `750,000`                         |
| **Internal Maintenance**      | Matches max faults per type                   | Matches max faults per type                |
| **External Maintenance**      | Zero                                          | Zero                                       |
| **Internal Maintenance Decision** | Yes                                       | Yes                                        |

---


The **MaintenanceStrategies_revenue** approach provides a more comprehensive analysis by considering the impact of downtime on revenue. This results in varying revenues and downtime across fault types but consistently selects internal maintenance as the optimal strategy. In contrast, the **MaintenanceStrategies_cost** approach focuses solely on minimizing costs and does not account for operational downtime or revenue, leading to the same internal maintenance decisions but without insights into downtime or revenue loss. Both strategies highlight the fixed cost advantage and resilience of internal maintenance but differ in their emphasis and outcomes.


### With Preventation Maintenance (Resulting in 60% reduction in Faults)

In [26]:
# Define cost parameters
cost_internal = 750000
cost_external_normal = 50000
cost_external_high_demand = 150000
prevention_effect = 0.6


# Dictionary to store results
revenue_results = []

# Iterate through each fault type in total_fault_counts
for fault_type, max_faults in total_fault_counts.items():
    # Initialize the MaintenanceStrategies_revenue class
    maintenance_strategy = MaintenanceStrategies_revenue(
        max_faults=max_faults,
        high_seasonality=high_seasonality,
        cost_internal=cost_internal,
        cost_external_normal=cost_external_normal,
        cost_external_high_demand=cost_external_high_demand,
        prevention_effect=prevention_effect,
        repair_time_external_hr=repair_time_external_hr,
        repair_time_internal_hr=repair_time_internal_hr,
        operation_time_hr=total_operation_hr,
        max_revenue=max_generated_revenue
    )
    
    # Run the maintenance decision method
    (
        total_revenue,
        total_downtime,
        total_cost,
        total_internal_maintenance,
        total_external_normal_maintenance,
        total_external_high_maintenance
    ) = maintenance_strategy.maintenance_decision()
    
    # Determine if internal maintenance was chosen (yes/no)
    internal_maintenance_decision = "Yes" if total_internal_maintenance > 0 else "No"
    
    # Append results for this fault type
    revenue_results.append({
        "Fault Type": fault_type,
        "Max Faults": max_faults,
        "Total Revenue": total_revenue,
        "Total Downtime (hrs)": total_downtime,
        "Total Cost": total_cost,
        "Internal Maintenance": total_internal_maintenance,
        "External Maintenance (Normal)": total_external_normal_maintenance,
        "External Maintenance (High-Demand)": total_external_high_maintenance,
        "Internal Maintenance Decision": internal_maintenance_decision
    })

# Convert the results to a DataFrame for easy visualization
revenue_results_df = pd.DataFrame(revenue_results)

# Display the DataFrame
revenue_results_df


,Fault Type,Max Faults,Total Revenue,Total Downtime (hrs),Total Cost,Internal Maintenance,External Maintenance (Normal),External Maintenance (High-Demand),Internal Maintenance Decision
0,FF,254,2.000594e+08,816,750000,102,0,0,Yes
1,EF,174,2.073326e+08,560,750000,70,0,0,Yes
2,AF,62,2.175606e+08,200,750000,25,0,0,Yes
3,GF,43,2.193789e+08,136,750000,17,0,0,Yes
4,MF,20,2.217291e+08,48,900000,0,3,5,No



1. **Internal Maintenance Decision**:
   - For all fault types (`FF`, `EF`, `AF`, `GF`), the strategy resulted in an **Internal Maintenance Decision** ("Yes") except for `MF`. In `MF`, external maintenance was selected due to its lower cost, reflecting the flexibility in the strategy.
   
2. **Revenue Impact**:
   - Total revenue improved significantly for all fault types compared to scenarios without preventative maintenance due to reduced downtime.
   - Downtime decreased as faults were reduced by 60%, directly enhancing operational efficiency.

3. **Total Cost**:
   - Internal maintenance cost remained constant for most fault types (`750,000`), except for `MF`, where the external maintenance cost (`900,000`) was higher due to the specific decision.



#### Comparison with MaintenanceStrategies_revenue Without Preventative Maintenance:
| Metric                        | MaintenanceStrategies_revenue (With Prevention) | MaintenanceStrategies_revenue (No Prevention) |
|-------------------------------|------------------------------------------------|----------------------------------------------|
| **Total Revenue**             | Higher due to reduced faults and downtime      | Lower due to increased faults and downtime  |
| **Total Downtime (hrs)**      | Reduced significantly                          | High, proportional to fault count           |
| **Total Cost**                | Adaptive to the chosen strategy                | Fixed `750,000`                             |
| **Internal Maintenance Decision** | Mostly Yes, selective external (e.g., `MF`)  | Yes for all                                  |
| **External Maintenance**      | Applied selectively                            | Zero                                         |


1. **Preventative Maintenance Benefits**:
   - Preventative maintenance led to a 60% reduction in faults, which significantly decreased downtime and increased revenue in the revenue-based strategy.
   - The flexibility in `MaintenanceStrategies_revenue` allowed for external maintenance selection (`MF`), showcasing adaptability to cost and downtime trade-offs.

2. **Comparison with MaintenanceStrategies_cost**:
   - The cost-focused strategy rigidly selects internal maintenance due to its fixed cost, while the revenue-based strategy adapts decisions to maximize revenue.

3. **Comparison with No Prevention**:
   - Without preventative maintenance, fault counts, downtime, and operational inefficiencies were higher, leading to lower revenue. Preventative maintenance effectively mitigated these challenges.



Preventative maintenance enhances revenue, reduces downtime, and supports more strategic decision-making in `MaintenanceStrategies_revenue`. Compared to `MaintenanceStrategies_cost`, the revenue-based approach provides greater flexibility, particularly when preventative measures are implemented. However, in scenarios without preventative maintenance, both strategies consistently select internal maintenance due to its cost-effectiveness.


### General Summary and Decision-Making Strategies for Maintenance Selection

#### Overview
The analysis across both **MaintenanceStrategies_cost** (minimizing cost) and **MaintenanceStrategies_revenue** (maximizing revenue), under scenarios with and without preventative maintenance, provides key insights into how decisions are made regarding internal vs. external maintenance. The results consistently highlight the trade-offs between cost, downtime, and revenue under different conditions.

---

### **Key Findings**
#### 1. **MaintenanceStrategies_cost (Minimizing Cost)**
- **Without Preventative Maintenance**:
  - All fault types consistently selected **internal maintenance** due to its fixed cost (`750,000`), which was lower than the total cost of external maintenance.
  - External maintenance was not chosen in any scenario as its cost (variable per fault and seasonality) exceeded the internal fixed cost.

- **With Preventative Maintenance (60% Fault Reduction)**:
  - Internal maintenance remained the optimal choice for all fault types as the reduced fault count further validated the fixed cost efficiency of internal maintenance (`750,000`).
  - No external maintenance was selected, as it was consistently more expensive.

#### 2. **MaintenanceStrategies_revenue (Maximizing Revenue)**
- **Without Preventative Maintenance**:
  - All fault types selected **internal maintenance**, as it minimized downtime and maximized total revenue compared to external maintenance.
  - Downtime and revenue losses were significant due to the higher fault counts, but internal maintenance consistently outperformed external maintenance.

- **With Preventative Maintenance (60% Fault Reduction)**:
  - For most fault types, **internal maintenance** was selected as it provided the highest revenue due to reduced downtime.
  - However, for the `MF` fault type, **external maintenance** was chosen as it provided slightly better revenue by balancing repair time and costs.
  - The strategy exhibited flexibility, adapting to fault type-specific trade-offs between cost, downtime, and revenue.

---

### **Comparison Between Strategies**
| Metric                        | MaintenanceStrategies_cost         | MaintenanceStrategies_revenue     |
|-------------------------------|------------------------------------|------------------------------------|
| **Focus**                     | Cost Minimization                 | Revenue Maximization              |
| **Flexibility**               | Rigid: Always selects internal    | Adaptive: Balances internal vs. external |
| **Impact of Preventative Maintenance** | Reduces fault count but does not change decision (internal preferred) | Reduces fault count, improves revenue, and adapts decisions based on trade-offs |
| **Downtime Consideration**    | Not considered                   | Directly influences revenue and decisions |
| **External Maintenance**      | Never selected                   | Selectively chosen for specific cases (e.g., `MF`) |

---

### **Decision-Making Strategies**
1. **Cost-Driven Organizations**:
   - Use **MaintenanceStrategies_cost** to minimize costs. 
   - Select **internal maintenance** when its fixed cost is consistently lower than the variable costs of external maintenance, regardless of fault count or prevention effects.
   - Preventative maintenance is still recommended to reduce fault counts, even though it doesn't change the decision.

2. **Revenue-Driven Organizations**:
   - Use **MaintenanceStrategies_revenue** to maximize revenue by balancing downtime, repair times, and fault count.
   - Select **internal maintenance** when downtime-related revenue losses are minimal, but be open to **external maintenance** for fault types where it offers higher overall revenue.
   - Implement preventative maintenance to reduce downtime and increase operational revenue.

---

### **General Conclusion**
- **Preventative Maintenance**: Always beneficial, as it reduces fault counts and downtime, improving both cost and revenue outcomes.
- **Internal Maintenance**: The preferred choice in most scenarios due to its fixed cost efficiency and minimal downtime impact.
- **External Maintenance**: Viable for specific fault types or scenarios where the trade-offs (e.g., reduced repair time or seasonality effects) lead to better overall outcomes, especially under a revenue-maximizing strategy.
- **Strategic Flexibility**: Revenue-driven strategies provide better adaptability and decision-making based on operational priorities, while cost-driven strategies are more rigid and focused solely on minimizing expenses.

Organizations should evaluate their operational goals (cost vs. revenue focus), leverage preventative maintenance, and choose a strategy that aligns with their objectives and resources.


### Assumptions of the Analysis

The analysis conducted in this notebook relies on several key assumptions. These assumptions impact the thresholds and decisions illustrated in the sensitivity analysis graphs. If these assumptions change, the outcomes and interpretations could vary significantly.

1. **Fixed Cost of Maintenance**:
   - The cost of **internal maintenance** is treated as a fixed expense, irrespective of the fault count.
   - External maintenance costs are proportional to the number of faults and are split between normal and high-demand season costs.

   **Impact**: If these costs were dynamic (e.g., bulk discounts for external maintenance or variable internal costs), the decision thresholds would shift.

---

 2. **Repair Downtime**:
   - **Internal Maintenance**:
     - Assumes a longer downtime due to in-house procedures and resource constraints.
   - **External Maintenance**:
     - Assumes a shorter downtime due to specialization or external expertise.

   **Impact**: Changes in the relative downtime between internal and external strategies would alter the preference for each strategy, particularly in the **Revenue Maximization** analysis where downtime directly affects revenue.

---

3. **Preventative Maintenance Cost is Missing**:
   - Preventative maintenance is assumed to reduce faults by a fixed proportion (60% in the scenarios analyzed), but it has no impact in the cost.

   **Impact**: If the Preventative maintenance has a signifcant cost it would have impact both strategies in this analysis as it would affect cost and reveune. 

---
 4. **Fault Distribution**:
   - Faults are distributed between high-demand and normal seasons based on a fixed `high_seasonality` proportion.
   - No additional complexity, such as varying fault severity or clustering, is modeled.

   **Impact**: Changes in fault distribution or inclusion of fault severity levels would require adjustments to external and internal maintenance strategies, potentially affecting costs and preferences.

---

 5. **Revenue Impact of Downtime**:
   - In the **Revenue Maximization** strategy, downtime directly reduces revenue in proportion to the total operation time and maximum possible revenue.
   - Assumes that downtime scales linearly with the number of faults addressed.

   **Impact**: If downtime has nonlinear effects (e.g., compounding losses during peak demand periods), the thresholds and preferences would shift.

---

6. **Focus on Binary Maintenance Decisions**:
   - Decisions are treated as binary (internal vs. external), with no intermediate options or shared responsibilities.

   **Impact**: Introducing mixed strategies (e.g., partial outsourcing) would make the decision-making process more complex and nuanced.

---


These assumptions simplify the analysis but also constrain the generalizability of the results. Future iterations of the analysis could incorporate dynamic costs, variable downtimes, or hybrid strategies to better reflect real-world scenarios. Acknowledging these assumptions highlights the sensitivity of the decisions to changes in key parameters and the importance of tailoring the analysis to specific operational contexts.
